In [1]:
import json
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import optuna
import gc
from pathlib import Path
from PIL import Image
from sklearn.model_selection import KFold

#pytorch stuff
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
#### TRAINING CONFIGURATION ####

pretrained = True
frozen = False
num_folds = 3
max_epochs = 50
patience = 10
checkpoint_every = 5
resume = False

In [3]:
#### regime configuration

regime_config = {
    (False, False): {"label": "ML_scratch", "study_name": "MLResnet18_no_weights_BCE", "storage": "sqlite:///optuna_studies/optuna_MLresnet18_no_weights_BCE.db"},
    (True, False): {"label": "ML_pretrained_finetuned", "study_name": "MLResnet18_pretrained_BCEloss", "storage": "sqlite:///optuna_studies/optuna_MLresnet18_pretrained_BCE.db"},
    (True, True): {"label": "ML_pretrained_frozen", "study_name":"MLResnet18_pretrained_frozen_BCE", "storage": "sqlite:///optuna_studies/optuna_MLresnet18_pretrained_frozen_BCE.db"},
}
regime = regime_config[(pretrained, frozen)]
regime_label = regime["label"]

print("Regime:", regime_label)
print("pretrained:", pretrained)
print("frozen:", frozen)

Regime: ML_pretrained_finetuned
pretrained: True
frozen: False


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Running on GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Running on CPU")

Running on GPU: Tesla T4


In [5]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


## Load the dataset

In [6]:
#### DATASET PREPROCESSING #####
img_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
map_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor()]) #transforms.Normalize((0.5,), (0.5,))])

In [7]:
#### CREATE DATASET ####
class create_dataset(Dataset):
    """ Custom dataset for loading input images and their optional ground-truth saliency maps. """
    def __init__(self, image_paths, map_paths=None, image_transform=img_transf, map_transform=map_transf):
        self.image_paths = image_paths
        self.map_paths = map_paths
        self.image_transform = image_transform
        self.map_transform = map_transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        
        image = Image.open(self.image_paths[idx]).convert("RGB")

        if self.image_transform is not None:
            image = self.image_transform(image)

        # no groundtruth
        if self.map_paths is None:
            return image

        saliency_map = Image.open(self.map_paths[idx]).convert("L")

        if self.map_transform is not None:
            saliency_map = self.map_transform(saliency_map)

        return image, saliency_map

def create_path(path):
    """ This function return a sorted list of path objects contained in the given directory. """
    return sorted(Path(path).iterdir())

In [8]:
#### PATHS ####
train_images_path = create_path("dataset/images/train/")
train_maps_path = create_path("dataset/maps/train/")

val_images_path = create_path("dataset/images/val/")
val_maps_path = create_path("dataset/maps/val/")

In [9]:
#training set 10k
training_set = create_dataset(train_images_path, train_maps_path)

#validation set 5k
full_validation_set = create_dataset(val_images_path, val_maps_path)

#seed for repeating the same experiment with all models, namely the same sets
rng = np.random.default_rng(42)
indices = rng.permutation(len(full_validation_set))
split = int(len(indices) * 0.6)

#3k validation
validation_set = Subset(full_validation_set, indices[:split].tolist())

#2k test
test_set = Subset(full_validation_set, indices[split:].tolist())

# 3-fold CV is performed only on development data
development_set = ConcatDataset([training_set, validation_set])

print("Dev. samples:", len(development_set))
print("Indep. test samples:", len(test_set))


Dev. samples: 13000
Indep. test samples: 2000


## Model

In [8]:
#This is basically the same encoder used in SSResNet-18, the difference is that we extract the features at different depths in the forward pass
class MLresnet18_encoder(nn.Module):
    """Encoder based on resnet18.
       Features are extracted at three different levels.
       The encoder can be frozen by setting frozen=True.
       Pretrained ImageNet weights can be used by setting pretrained=True. """

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()

        #weights choice
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
        else:
            weights = None

        resnet = resnet18(weights=weights)

        #initial layer
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        #residual layers
        self.layer1 = resnet.layer1    # H/4 x W/4
        self.layer2 = resnet.layer2    # H/8 x W/8
        self.layer3 = resnet.layer3    # H/16 x W/16
        self.layer4 = resnet.layer4    # H/16 x W/16
        self.layer4[0].conv1.stride = (1, 1) # set stride 1 instead of 2 (originally was /32, but i set the stride 1)
        self.layer4[0].downsample[0].stride = (1, 1) # skip connection with stride 1 instead of 2


        self.frozen = frozen #frozen state for the encoder
        if self.frozen:
            self.freeze()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        feature1 = self.layer1(x)
        feature2 = self.layer2(feature1)
        layer3_output = self.layer3(feature2)
        feature3 = self.layer4(layer3_output)
        return feature1, feature2, feature3

    def freeze(self):
        """ Freeze all encoder parameters and set encoder in eval mode. """
        self.frozen = True
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def unfreeze(self):
        """ unfreeze encoder parameters."""
        self.frozen = False
        for p in self.parameters():
            p.requires_grad = True
        self.train()

    def train(self, mode=True):
        """ Keep the encoder in eval mode when is frozen to prevent BN updates."""
        super().train(mode)
        if self.frozen:
            super().train(False)
        return self

In [9]:
class decoder(nn.Module):
    def __init__(self):
        super().__init__()

        #1x1 channel projections
        #they adapt resnet channels to the dimensionalities used by the decoders (it's more clear in the forward pass)
        self.projection1 = nn.Conv2d(64, 256, kernel_size=1)
        self.projection2 = nn.Conv2d(128, 512, kernel_size=1)
        self.projection3 = nn.Conv2d(512, 512, kernel_size=1)

        #decoder 1: H/4 -> H
        self.decoder1_up_layer1 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder1_up_layer2 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier1 = nn.Conv2d(64, 1, kernel_size=1)

        #decoder 2: H/8 -> H
        self.decoder2_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer2 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer3 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier2 = nn.Conv2d(64, 1, kernel_size=1)

        #decoder 3: H/16 -> H
        self.decoder3_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 512, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer2 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer3 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer4 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier3 = nn.Conv2d(64, 1, kernel_size=1)

        #learned fusion of the three saliency logits
        self.fusion = nn.Conv2d(3, 1, kernel_size=1)


    def forward(self, features):
        feature1, feature2, feature3 = features

        #channel projection
        feature1 = self.projection1(feature1)    # 64 -> 256
        feature2 = self.projection2(feature2)    # 128 -> 512
        feature3 = self.projection3(feature3)    # 512 -> 512

        #decoder 1
        x1 = self.decoder1_up_layer1(feature1)
        x1 = self.decoder1_up_layer2(x1)
        logit1 = self.classifier1(x1)

        #decoder 2
        x2 = self.decoder2_up_layer1(feature2)
        x2 = self.decoder2_up_layer2(x2)
        x2 = self.decoder2_up_layer3(x2)
        logit2 = self.classifier2(x2)

        #decoder 3
        x3 = self.decoder3_up_layer1(feature3)
        x3 = self.decoder3_up_layer2(x3)
        x3 = self.decoder3_up_layer3(x3)
        x3 = self.decoder3_up_layer4(x3)
        logit3 = self.classifier3(x3)

        #fusion
        multi_level_logits = torch.cat((logit1, logit2, logit3), dim=1)
        fused_logit = self.fusion(multi_level_logits)

        return {"level1": logit1, "level2": logit2, "level3": logit3, "fused": fused_logit}

In [ ]:
### DS loss implemented as in the reference paper
class deep_supervision_loss(nn.Module):
    """Deep-supervision BCE loss.
       Training objective:
           total_loss = (BCE_level1 + BCE_level2 + BCE_level3) / 3 + BCE_fusion
       The model returns logits. """

    def __init__(self):
        super().__init__()
        self.BCE = nn.BCEWithLogitsLoss()

    def forward(self, outputs, maps):
        loss1 = self.BCE(outputs["level1"], maps)
        loss2 = self.BCE(outputs["level2"], maps)
        loss3 = self.BCE(outputs["level3"], maps)
        fusion_loss = self.BCE(outputs["fused"], maps)
        intermediate_loss = (loss1 + loss2 + loss3) / 3
        total_loss = (intermediate_loss + fusion_loss)
        return total_loss

In [10]:
class encoder_decoder(nn.Module):
    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        self.encoder = MLresnet18_encoder(pretrained=pretrained, frozen=frozen)
        self.decoder = decoder()

    def forward(self, x):
        # skip gradient tracking when encoder is frozen
        if self.encoder.frozen:
            with torch.no_grad():
                features = self.encoder(x)
        else:
            features = self.encoder(x)

        output = self.decoder(features)
        return output

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()

## Training

In [11]:
class trainer:
    def __init__(self, model, optimizer, loss_function, device):
        
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.device = device
        self.use_amp = (self.device.type == "cuda")
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)


    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        total_samples = 0

        for images, maps in train_loader:
            images = images.to(self.device, non_blocking=True)
            maps = maps.to(self.device, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):

                outputs = self.model(images)
                # DS loss
                loss = self.loss_function(outputs, maps)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            batch_size = images.size(0)
            total_loss += (loss.detach().float().item() * batch_size)
            total_samples += batch_size

        return (total_loss / total_samples)


    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0
        total_level1_BCE = 0
        total_level2_BCE = 0
        total_level3_BCE = 0
        total_fused_BCE = 0
        total_samples = 0

        BCE = nn.BCEWithLogitsLoss()

        with torch.no_grad():
            for images, maps in val_loader:
                images = images.to(self.device, non_blocking=True)
                maps = maps.to(self.device, non_blocking=True)

                with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):

                    outputs = self.model(images)
                    loss = self.loss_function(outputs, maps)

                    # BCE of each individual output
                    level1_BCE = BCE(outputs["level1"], maps)
                    level2_BCE = BCE(outputs["level2"], maps)
                    level3_BCE = BCE(outputs["level3"], maps)
                    #final BCE
                    fused_BCE = BCE(outputs["fused"], maps)
                    
                batch_size = images.size(0)
                total_loss += (loss.detach().float().item() * batch_size)
                total_level1_BCE += (level1_BCE.detach().float().item() * batch_size)
                total_level2_BCE += (level2_BCE.detach().float().item() * batch_size)
                total_level3_BCE += (level3_BCE.detach().float().item() * batch_size)
                total_fused_BCE += (fused_BCE.detach().float().item() * batch_size)
                total_samples += batch_size


        metrics = {"val_loss": total_loss / total_samples, "level1_BCE": total_level1_BCE / total_samples, "level2_BCE": total_level2_BCE / total_samples, 
                   "level3_BCE": total_level3_BCE / total_samples, "fused_BCE": total_fused_BCE / total_samples}
        
        return metrics

    
    def fit(self, train_loader, val_loader,epochs, fold_dir, checkpoint_every=5, patience=10, resume_checkpoint=None):
        fold_dir = Path(fold_dir)
        fold_dir.mkdir(parents=True, exist_ok=True)

        history = {k: [] for k in ["train_loss", "val_loss", "level1_BCE", "level2_BCE","level3_BCE","fused_BCE"]}
        best_fused_BCE = float("inf")
        best_epoch = None
        best_model_state = None
        start_epoch = 0
        epochs_without_improvement = 0

        if resume_checkpoint is not None:
            checkpoint = torch.load(resume_checkpoint, map_location=self.device)

            self.model.load_state_dict(checkpoint["model_state_dict"])
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])

            start_epoch = checkpoint["epoch"]
            history = checkpoint["history"]
            best_fused_BCE = checkpoint["best_fused_BCE"]
            best_epoch = checkpoint["best_epoch"]
            epochs_without_improvement = checkpoint["epochs_without_improvement"]
            print(f"Resuming from epoch {start_epoch}", flush=True)

        for epoch in range(start_epoch, epochs):
            if epochs_without_improvement >= patience:
                break
            epoch_start = time.perf_counter()

            train_loss = self.train_epoch(train_loader)
            val_metrics = self.validate(val_loader)
            val_fused_BCE = val_metrics["fused_BCE"]

            history["train_loss"].append(train_loss)
            
            for key, value in val_metrics.items():
                history[key].append(value)

            if val_fused_BCE < best_fused_BCE:

                best_fused_BCE = val_fused_BCE
                best_epoch = epoch + 1
            
                epochs_without_improvement = 0
            
                best_model_state = {key: value.detach().cpu().clone() for key, value in self.model.state_dict().items()}
            
                torch.save({"epoch": best_epoch, "model_state_dict": best_model_state, "best_fused_BCE": best_fused_BCE, "selection_metric": "fused_BCE", 
                            "metrics": val_metrics, "history": history}, fold_dir / "best_model.pth")
            else:
                epochs_without_improvement += 1

            epoch_time = time.perf_counter() - epoch_start

            print(f"epoch {epoch + 1}/{epochs} | train total loss: {train_loss:.6f} | val fused BCE: {val_fused_BCE:.6f} | best loss: {best_fused_BCE:.6f} |" 
                  f"best epoch: {best_epoch} |  patience: {epochs_without_improvement}/{patience} | time: {epoch_time:.1f}s", flush=True)

            if (epoch + 1) % checkpoint_every == 0:
                checkpoint_path = (fold_dir /f"checkpoint_epoch_{epoch + 1}.pth")

                torch.save({"epoch": epoch + 1, "model_state_dict": self.model.state_dict(), "optimizer_state_dict": self.optimizer.state_dict(), "scaler_state_dict": self.scaler.state_dict(),
                            "best_fused_BCE": best_fused_BCE, "best_epoch": best_epoch, "epochs_without_improvement": epochs_without_improvement, "history": history}, checkpoint_path)

                print(f"Checkpoint saved: {checkpoint_path}", flush=True)
                
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1} | best epoch: {best_epoch} | best val BCE: {best_fused_BCE:.6f}", flush=True)
                break
                
        best_path = fold_dir / "best_model.pth"

        if best_path.exists():
            best_checkpoint = torch.load( best_path, map_location=self.device )
            self.model.load_state_dict(best_checkpoint["model_state_dict"])
            best_fused_BCE = best_checkpoint["best_fused_BCE"]
            best_epoch = best_checkpoint["epoch"]

        elif best_model_state is not None:
            self.model.load_state_dict(best_model_state)

        return best_fused_BCE, best_epoch, history


## Load Optuna best hyperparameters

In [ ]:
study = optuna.load_study(study_name=regime["study_name"], storage=regime["storage"])

best_params = study.best_params

print("Best trial:", study.best_trial.number)
print("Best validation BCE:", study.best_value)
print("Best parameters:")

for key, value in best_params.items():
    print(f"  {key}: {value}")

In [ ]:
def create_optimizer(model, best_params, frozen):
    optimizer_name = best_params["optimizer"]

    if frozen:
        parameter_groups = [{"params": model.decoder.parameters(), "lr": best_params["lr_decoder"]}]
    else:
        parameter_groups = [{"params": model.encoder.parameters(), "lr": best_params["lr_encoder"]},
                            {"params": model.decoder.parameters(), "lr": best_params["lr_decoder"]}]

    optimizer_options = {"weight_decay": best_params["weight_decay"]}

    if optimizer_name in ["SGD", "RMSprop"]:
        optimizer_options["momentum"] = best_params["momentum"]
    else:
        optimizer_options["betas"] = (best_params["beta1"], best_params["beta2"])

    optimizer_class = getattr(torch.optim, optimizer_name)

    return optimizer_class(parameter_groups, **optimizer_options)

## K-fold CV

In [ ]:
batch_size = best_params["batch_size"]

loader_options = dict(num_workers=12, pin_memory=(device.type == "cuda"), persistent_workers=True, prefetch_factor=4)

kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)

run_dir = Path("training_runs") / regime_label
run_dir.mkdir(parents=True, exist_ok=True)

fold_results = []
fold_histories = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(len(development_set))),start=1):

    print("\n" + "=" * 84)
    print(f"FOLD {fold}/{num_folds}")
    print("=" * 84, flush=True)

    fold_seed = seed + fold

    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)


    fold_train_set = Subset(development_set, train_idx.tolist())
    fold_val_set = Subset(development_set, val_idx.tolist())

    train_loader = DataLoader(fold_train_set, batch_size=batch_size, shuffle=True, **loader_options)
    val_loader = DataLoader(fold_val_set,batch_size=batch_size, shuffle=False, **loader_options)

    #new model
    model = encoder_decoder(pretrained=pretrained, frozen=frozen).to(device)
    optimizer = create_optimizer(model, best_params, frozen)

    loss_function = deep_supervision_loss()

    model_trainer = trainer(model=model, optimizer=optimizer, loss_function=loss_function, device=device)

    fold_dir = run_dir / f"fold_{fold}"
    fold_dir.mkdir(parents=True,exist_ok=True)

    resume_checkpoint = None

    if resume:
        checkpoints = sorted(fold_dir.glob("checkpoint_epoch_*.pth"), key=lambda p: int(p.stem.split("_")[-1]))
        if checkpoints:
            resume_checkpoint = checkpoints[-1]
            print("Latest checkpoint:", resume_checkpoint,flush=True)

    best_fused_BCE, best_epoch, history = (model_trainer.fit(train_loader=train_loader, val_loader=val_loader, epochs=max_epochs, fold_dir=fold_dir, 
                                                             checkpoint_every=checkpoint_every, patience=patience, resume_checkpoint=resume_checkpoint))

    fold_results.append({"fold": fold, "best_epoch": best_epoch, "best_val_bce": best_fused_BCE})
    fold_histories.append(history)


    with open(fold_dir / "history.json", "w") as f:
        json.dump(history, f)

    print(f"Fold {fold} completed | best epoch={best_epoch} | best fused BCE={best_fused_BCE:.6f}", flush=True)

    del train_loader, val_loader
    del model, optimizer, model_trainer
    
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## CV summary

In [ ]:
fold_bce = np.array([result["best_val_bce"] for result in fold_results])

print(f"\nk-fold CV completed")

for result in fold_results:
    print(f"Fold {result['fold']} | best epoch={result['best_epoch']} | best fused BCE={result['best_val_bce']:.6f}")

print(f"\nMean validation fused BCE: {fold_bce.mean():.6f}")
print(f"Std validation fused BCE: {fold_bce.std(ddof=1):.6f}")

with open(run_dir / "cv_results.json", "w") as f:
    json.dump({"regime": regime_label, "pretrained": pretrained, "frozen": frozen, "n_splits": num_folds, "max_epochs": max_epochs, "checkpoint_every": checkpoint_every, 
            "best_optuna_params": best_params, "fold_results": fold_results, "mean_val_bce": float(fold_bce.mean()),"std_val_bce": float(fold_bce.std(ddof=1))}, f, indent=2)